# Titanic EDA 

**Goal:** Explore the Titanic dataset, understand what affects survival, and produce clean visualisations.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make plots look clean
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully!')

---
## Load the data


In [ ]:

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

print(f'Dataset shape: {df.shape}')  # (rows, columns)
print(f'Rows: {df.shape[0]}, Columns: {df.shape[1]}')

---
## — First look at the data


In [ ]:
# See the first 5 rows
df.head()

In [ ]:
# Column names and data types
print(df.dtypes)

# Key column meanings:
# Survived  -> 0 = died, 1 = survived
# Pclass    -> ticket class: 1 = first, 2 = second, 3 = third
# Sex       -> male or female
# Age       -> passenger age in years
# SibSp     -> number of siblings/spouses aboard
# Parch     -> number of parents/children aboard
# Fare      -> ticket price
# Embarked  -> port of embarkation (C, Q, S)

In [ ]:
# Summary statistics for numeric columns
df.describe()

---
## Check for missing values

Real data is messy. Missing values must be handled before analysis or modelling.

In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
})

# Only show columns that actually have missing values
print(missing_df[missing_df['Missing Count'] > 0])

In [ ]:
# Visualise missing values as a heatmap
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='viridis')
plt.title('Missing values (yellow = missing)')
plt.tight_layout()
plt.show()

# INSIGHT: Age has ~20% missing. Cabin has ~77% missing (too many to use).
# Embarked has only 2 missing — easy to fix.

---
##  Clean the data

We'll handle the three columns with missing values.

In [ ]:
# 1. Age: fill missing with the median age (robust to outliers)
median_age = df['Age'].median()
df['Age'].fillna(median_age, inplace=True)
print(f'Age missing values filled with median: {median_age}')

# 2. Embarked: fill 2 missing rows with the most common port
most_common_port = df['Embarked'].mode()[0]
df['Embarked'].fillna(most_common_port, inplace=True)
print(f'Embarked missing values filled with: {most_common_port}')

# 3. Cabin: drop it — 77% missing is too much to salvage usefully
df.drop(columns=['Cabin'], inplace=True)
print('Cabin column dropped.')

# Verify no more missing values in key columns
print('\nRemaining missing values:')
print(df.isnull().sum()[df.isnull().sum() > 0])

---
## Explore survival rate overall

Before diving into features, understand the baseline: what % of passengers survived?

In [ ]:
survival_rate = df['Survived'].mean() * 100
print(f'Overall survival rate: {survival_rate:.1f}%')
print(f'Survived: {df["Survived"].sum()} passengers')
print(f'Died:     {(df["Survived"] == 0).sum()} passengers')

# Plot
fig, ax = plt.subplots(figsize=(5, 4))
counts = df['Survived'].value_counts()
bars = ax.bar(['Died (0)', 'Survived (1)'], counts, color=['#E24B4A', '#1D9E75'], width=0.5)

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(count), ha='center', va='bottom', fontweight='bold')

ax.set_title('Survival count — Titanic dataset')
ax.set_ylabel('Number of passengers')
plt.tight_layout()
plt.show()

---
## Insight 1: Survival by Sex

> "Women and children first" — let's verify this in the data.

In [ ]:
survival_by_sex = df.groupby('Sex')['Survived'].mean() * 100
print('Survival rate by sex:')
print(survival_by_sex.round(1))

fig, ax = plt.subplots(figsize=(5, 4))
colors = ['#378ADD', '#D4537E']
bars = ax.bar(survival_by_sex.index, survival_by_sex.values, color=colors, width=0.4)

for bar, val in zip(bars, survival_by_sex.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')

ax.set_ylim(0, 100)
ax.set_title('Survival rate by sex')
ax.set_ylabel('Survival rate (%)')
plt.tight_layout()
plt.show()

# INSIGHT: Female survival rate is ~74%. Male is ~19%.
# Sex is one of the strongest predictors of survival.

---
## Insight 2: Survival by Passenger Class

Did wealth affect your chances of survival?

In [ ]:
survival_by_class = df.groupby('Pclass')['Survived'].mean() * 100
print('Survival rate by passenger class:')
print(survival_by_class.round(1))

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(
    ['1st class', '2nd class', '3rd class'],
    survival_by_class.values,
    color=['#534AB7', '#7F77DD', '#AFA9EC'],
    width=0.4
)

for bar, val in zip(bars, survival_by_class.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.1f}%', ha='center', va='bottom', fontweight='bold')

ax.set_ylim(0, 100)
ax.set_title('Survival rate by passenger class')
ax.set_ylabel('Survival rate (%)')
plt.tight_layout()
plt.show()

# INSIGHT: 1st class ~63%, 2nd class ~47%, 3rd class ~24%.
# Wealthier passengers had significantly better survival odds.

---
## Insight 3: Age distribution by survival

Were younger passengers more likely to survive?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

survived = df[df['Survived'] == 1]['Age']
died = df[df['Survived'] == 0]['Age']

ax.hist(died, bins=30, alpha=0.6, color='#E24B4A', label='Died', edgecolor='white')
ax.hist(survived, bins=30, alpha=0.6, color='#1D9E75', label='Survived', edgecolor='white')

ax.axvline(died.mean(), color='#A32D2D', linestyle='--', linewidth=1.5,
           label=f'Avg age died: {died.mean():.1f}')
ax.axvline(survived.mean(), color='#0F6E56', linestyle='--', linewidth=1.5,
           label=f'Avg age survived: {survived.mean():.1f}')

ax.set_title('Age distribution — survived vs died')
ax.set_xlabel('Age')
ax.set_ylabel('Number of passengers')
ax.legend()
plt.tight_layout()
plt.show()

# INSIGHT: Young children (under ~10) had notably higher survival.
# Average age of survivors is slightly lower than those who died.

---
## Insight 4: Survival by Class AND Sex combined

The real power of EDA is finding interactions between features.

In [ ]:
pivot = df.pivot_table(values='Survived', index='Pclass',
                       columns='Sex', aggfunc='mean') * 100
print('Survival rate (%) by class and sex:')
print(pivot.round(1))

fig, ax = plt.subplots(figsize=(7, 4))
pivot.plot(kind='bar', ax=ax, color=['#378ADD', '#D4537E'],
           width=0.5, edgecolor='white')

ax.set_xticklabels(['1st class', '2nd class', '3rd class'], rotation=0)
ax.set_ylim(0, 110)
ax.set_title('Survival rate by class and sex')
ax.set_ylabel('Survival rate (%)')
ax.set_xlabel('')
ax.legend(title='Sex')

for container in ax.containers:
    ax.bar_label(container, fmt='%.0f%%', padding=2, fontsize=9)

plt.tight_layout()
plt.show()

# INSIGHT: 1st class females had ~97% survival. 3rd class males had ~15%.
# The combination of sex + class is extremely predictive.

---
## Insight 5: Correlation heatmap

See how all numeric features relate to each other and to survival at once.

In [ ]:
# Select numeric columns only
numeric_cols = ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # hide upper triangle

sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.2f',
    cmap='RdYlGn',
    center=0,
    mask=mask,
    square=True,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation matrix — numeric features')
plt.tight_layout()
plt.show()

# INSIGHT:
# Pclass has a negative correlation with Survived (-0.34) — lower class = lower survival.
# Fare has a positive correlation (+0.26) — higher fare = higher survival.
# Age has a weak negative correlation (-0.07).

---
## Feature engineering

new features from existing ones — a key ML .

In [ ]:
# Family size: total people travelling together
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1  # +1 for the passenger themselves

# Is alone? (a simple binary feature)
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# Age group: bin continuous age into categories
df['AgeGroup'] = pd.cut(df['Age'],
                         bins=[0, 12, 18, 35, 60, 100],
                         labels=['Child', 'Teen', 'Young adult', 'Adult', 'Senior'])

# See survival rate by age group
print('Survival rate by age group:')
print((df.groupby('AgeGroup')['Survived'].mean() * 100).round(1))

# See survival rate by whether travelling alone
print('\nSurvival rate — alone vs with family:')
labels = {0: 'With family', 1: 'Alone'}
print((df.groupby('IsAlone')['Survived'].mean() * 100)
      .rename(index=labels).round(1))